In [ ]:
import geopandas as gpd
import polars as pl
import matplotlib.pyplot as plt
import us

buildings_path = "full_data_with_utility.csv"
states_path = "shapefiles/tl_2024_us_state.shp"
# IOU_path = "shapefiles/Gas Territories Fixed.shp"
IOU_path = "shapefiles/Electric Territory IOU.shp"
to_fips = us.states.mapping("abbr", "fips")

chosen_states = ["MA"]
# Load layers
buildings = pl.read_csv(buildings_path)
states    = gpd.read_file(states_path)
services  = gpd.read_file(IOU_path)

# Filter states and set CRS
states = states[states["STATEFP"].isin([to_fips[st] for st in chosen_states])]
buildings = buildings.filter(pl.col("in.state").is_in(chosen_states))
common_crs   = states.crs
services     = services.to_crs(common_crs)

# Prepare the IOU layer
if "Electric" in IOU_path:
    services = services[services["UtilitySta"].isin(chosen_states)]
else:
    services = services[services["State"].isin(chosen_states)]

# Build a GeoDataFrame of buildings
buildings = gpd.GeoDataFrame(
    buildings,
    geometry=gpd.points_from_xy(buildings["in.weather_file_longitude"], buildings["in.weather_file_latitude"]),
    crs="EPSG:4326"
)

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 10), dpi=100)

# State boundaries
states.boundary.plot(ax=ax, edgecolor="black", linewidth=1)

# Service territories
services.plot(
    ax=ax,
    column="Company",
    categorical=True,
    legend=True,
    linewidth=0.5,
    alpha=0.6,
    edgecolor="grey",
    cmap="tab20"
)

# Buildings plot
# group by geometry to count duplicates
counts = buildings.groupby(buildings.geometry).size().reset_index(name="count")
counts_gdf = gpd.GeoDataFrame(counts, geometry="geometry", crs=buildings.crs)

counts_gdf.plot(
    ax=ax,
    color="red",
    markersize=counts_gdf["count"]*0.5,
    label="Building Representative Points",
    alpha=0.5
)

ax.set_title(f"Buildings & Service Territories in {', '.join([st for st in chosen_states])} (n={sum(counts_gdf['count'])})")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()